In [1]:
import os
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()
tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY"))

def search_web(query: str, max_results: int = 3) -> list:
    response = tavily_client.search(query, max_results=max_results)
    return response.get("results")

search_web("Kipchoge's marathon world record")

[{'url': 'https://en.wikipedia.org/wiki/Eliud_Kipchoge',
  'title': 'Eliud Kipchoge - Wikipedia',
  'content': 'On 16 September, Kipchoge won the 2018 Berlin Marathon in a time of 2:01:39, breaking the previous world record by 1 minute and 18 seconds (2:02:57 set by fellow countryman Dennis Kimetto at the Berlin Marathon in 2014). It was the greatest improvement in a marathon world record time since 1967. He finished 4:43 min ahead of second-placed fellow Kenyan Amos Kipruto. The world record holder from 2013, Wilson Kipsang of Kenya, came in third at 2:06:48. It was the 11th world record set at the [...] On 25 September, Kipchoge won the Berlin Marathon decisively in a time of 2:01:09, beating by 30 seconds his own previous world record, which he set on the same course in 2018. With his fourth victory in Berlin, he equaled the record achievement of Haile Gebrselassie. He finished 4:49 min ahead of second-placed compatriot Mark Korir while Ethiopia\'s Tadu Abate took third place with a

In [2]:
def search_web(
    query: str,
    max_results: int = 5,
    topic: str = "general",
    time_range: str | None = None,
) -> list | str:
    """Search the web for the given query."""
    try:
        response = tavily_client.search(
            query,
            max_results=max_results,
            topic=topic,
            time_range=time_range,
        )
        return response.get("results")
    except Exception as e:
        return f"Error: Search failed - {e}"

In [3]:
import inspect

def example_tool(input_1:str, input_2:int=1):
    return

print(f"function name: {example_tool.__name__}")
print(f"function docstring: {example_tool.__doc__}")
print(f"function signature: {inspect.signature(example_tool)}")

#utility converting funct's signature to json schema for tool definitions
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    try:
        signature = inspect.signature(func)                     # A
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {e}"
        )

    parameters = {} #hold info for each param in function
    for param in signature.parameters.values():
        param_type = type_map.get(param.annotation, "string")    # B
        parameters[param.name] = {"type": param_type}            # B

    required = [
        param.name                                               # C
        for param in signature.parameters.values()               # C
        if param.default == inspect._empty                       # C
    ]

    return {
        "type": "object",
        "properties": parameters,
        "required": required,
    }

#build on prev utility: wrap input schema with funct name and docstr to create tool def
def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }


def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__,
        func.__doc__ or "",
        function_to_input_schema(func),
    )


search_tool_definition = function_to_tool_definition(search_web)
print(search_tool_definition)




function name: example_tool
function docstring: None
function signature: (input_1: str, input_2: int = 1)
{'type': 'function', 'function': {'name': 'search_web', 'description': 'Search the web for the given query.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'max_results': {'type': 'integer'}, 'topic': {'type': 'string'}, 'time_range': {'type': 'string'}}, 'required': ['query']}}}


In [4]:
#Run the requested function with arguments extracted from an LLM tool call or another syst
import json

def tool_execution(tool_box, tool_call):
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    tool_result = tool_box[function_name](**function_args)
    return tool_result

Agent loop, user input -> LLM, process tool calls, execute matching funcitons, update context

In [5]:
from litellm import completion

def simple_agent_loop(system_prompt, question):
    tools = [search_web]
    tool_box = {tool.__name__: tool for tool in tools}
    tool_definitions = [function_to_tool_definition(tool) for tool in tools]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    while True:
        response = completion(
            model="claude-sonnet-4-6",
            messages=messages,
            tools=tool_definitions,
        )
        assistant_message = response.choices[0].message

        if assistant_message.tool_calls: #if a model requested a tool
            messages.append(assistant_message)
            for tool_call in assistant_message.tool_calls:
                tool_result = tool_execution(tool_box, tool_call)
                messages.append({
                    "role": "tool",
                    "content": str(tool_result),
                    "tool_call_id": tool_call.id,
                })
        else:
            return assistant_message.content

#Test the agent loop
system_prompt = """You are a helpful assistant.
Use the search tool when you need current information."""

result = simple_agent_loop(
    system_prompt,
    "Who won the 2025 Nobel Prize in Physics?",
)
print(result)

The **2025 Nobel Prize in Physics** was awarded to **John Clarke**, **Michel H. Devoret**, and **John M. Martinis** for their groundbreaking work on the **discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit**. The prize was announced on **October 7, 2025**.


MCP connection

In [6]:
import os
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from dotenv import load_dotenv

load_dotenv()


async def open_tavily_session() -> tuple[ClientSession, AsyncExitStack]:
    """Start the Tavily MCP server once and return an open session."""
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY is not set - check your .env file")

    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "tavily-mcp@latest"],
        env={"TAVILY_API_KEY": api_key},
    )

    stack = AsyncExitStack()
    # A real file for server stderr - Jupyter's sys.stderr has no fileno() on Windows
    errlog = stack.enter_context(open("mcp_server.log", "w"))
    read_stream, write_stream = await stack.enter_async_context(stdio_client(server_params, errlog=errlog))
    session = await stack.enter_async_context(ClientSession(read_stream, write_stream))
    await session.initialize()
    return session, stack


async def mcp_tavily_search(session: ClientSession, query: str, max_results: int = 5) -> str:
    """Search the web via the Tavily MCP server and return the results as text."""
    result = await session.call_tool(
        "tavily_search",
        arguments={"query": query, "max_results": max_results},
    )
    # result.content is a list of content blocks; keep the text ones
    return "\n".join(block.text for block in result.content if block.type == "text")


In [7]:
session, stack = await open_tavily_session()

tools_result = await session.list_tools()
print("Available tools:", [tool.name for tool in tools_result.tools])

print(await mcp_tavily_search(session, "2025 Nobel Prize in Physics"))

Available tools: ['tavily_search', 'tavily_extract', 'tavily_crawl', 'tavily_map', 'tavily_research']
Detailed Results:

Title: Nobel Prize in Physics 2025
ID: c6752e-00
URL: https://www.nature.com/collections/fhdggghifa
Content: The Nobel Prize in Physics 2025 was awarded to John Clarke, Michel H. Devoret and John M. Martinis "for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit." The laureates performed experiments on electrical circuits in which they observed quantum mechanical phenomena such as tunnelling and quantised energy levels. Their work showed that quantum tunnelling is not confined to the quantum world, but it can also be reproduced in macroscopic [...] ## Highlights

### Groundbreaking quantum-tunnelling experiments win physics Nobel

John Clarke, Michel Devoret and John Martinis discovered quantum physics on a macroscopic scale, paving the way for quantum computing.

### Quantum Josephson junction circuits and the 

In [ ]:
#make function into MCP server
import os
from tavily import TavilyClient
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY"))

mcp = FastMCP("custom-tavily-search")


@mcp.tool()
def search_web(query: str, max_results: int = 5) -> str:
    """
    Search the web using Tavily API.

    Args:
        query: Search query string
        max_results: Maximum number of results to return (default: 5)

    Returns:
        Search results as formatted string
    """
    try:
        response = tavily_client.search(
            query,
            max_results=max_results,
        )
        results = response.get("results", [])
        return "\n\n".join(
            f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
            for r in results
        )
    except Exception as e:
        return f"Error searching web: {str(e)}"


if __name__ == "__main__":
    mcp.run(transport="stdio")



#connecting to custom MCP
server_params = StdioServerParameters(
    command="uv",
    args=["run", "tavily_mcp_server.py"],
    env={
        "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY"),
    },
)

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        # List available tools
        tools_result = await session.list_tools()
        print("Available tools:")
        for tool in tools_result.tools:
            print(f"  - {tool.name}: {tool.description}")